# Implementacion de Modelos de Machine Learning
A partir de los datos limpios realizados en la fase de limpieza de datos valga la redundancia, es posible comenzar a plantear la implementacion de modelos de machine learning de distintos tipos como regresiones, arboles o SVMs, sin embargo antes de ello es necesario realizar algunos ajustes a los datos limpios que se adecuen al proposito, problema o tarea a resolver que vayan a tener.

En este caso, se opto por que los modelos puedan predecir datos de casos de COVID-19 nuevos en una fecha determinada en China.

A lo largo de este documento se mostrara el proceso de adecuacion de los datos limpios para determinar la segregacion de los mismos en datos de entrenamiento y datos de prueba y eventualmente el aprendizaje de los modelos como tal.

# 1. Preparacion del Dataset Limpio

## 1.1. Inicializacion de Librerias y Datos
Se importan las siguientes librerias para lograr la implementacion de los modelos de regresion, arboles y SVMs:
- **os:** libreria del sistema operativo para el manejo de importacion de los archicos csv.
- **pandas:** libreria principal para el analisis de datos y consultas rapidas de las tablas importadas
- **numpy:** libreria para facilitar y optimizar operaciones matematicas y operaciones con listas con cantidades gigantes de valores.
- **scikit learn:** libreria para la importacion de modelos preimplementados de diferentes tipos y pipelines para unificar diferentes modelos y modificar sus caracteristicas.

In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler

Se inicializa la libreria de Google Colab para permitir acceso a Google Drive para el acceso a la carpeta compartida de ''Data Sciencie'' que contiene los archivos con extension .csv correspondiente al dataset limpio.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 1.2. Extraccion de Datos
Por medio del metodo *read_csv* de la libreria pandas se accede al directorio donde se ubica la carpeta "dataset_limpio" que contiene los archivos .csv limpiados con anterioridad por medio de la ruta absoluta: */content/drive/MyDrive/Data_Sciencie/covid_dataset/dataset_limpio* el cual corresponde a un directorio compartido en Google Drive.

De todos los archivos .csv limpios, se escoge exclusivamente los siguientes:
- **confirmed_cleaned.csv** almacenado en la variabla *confirmed*
- **deaths_cleaned.csv** almacenado en la variabla *deaths*
- **recovered_cleaned.csv** almacenado en la variabla *recovered*

Se descartan los archivos de **deaths_US_cleaned.csv** y **confirmed_US_cleaned** puesto que como grupo se definio que el alcance del modelo se centrara en la prediccion de casos nuevos confirmados de COVID-19 en un determinado dia en cualquier otro pais distinto a USA, ademas tambien se descarta el archivo **global_data_cleaned.csv**, ya que contiene informacion muy general que no se considero lo suficientemente util para el proposito mencionado.



In [ ]:
confirmed = pd.read_csv("/content/drive/MyDrive/Data Sciencie/covid_dataset/dataset_limpio/confirmed_cleaned.csv")
deaths = pd.read_csv("/content/drive/MyDrive/Data Sciencie/covid_dataset/dataset_limpio/deaths_cleaned.csv")
recovered = pd.read_csv("/content/drive/MyDrive/Data Sciencie/covid_dataset/dataset_limpio/recovered_cleaned.csv")

Es necesario dar forma a los contenidos extraidos anteriormente y almacenados en las variables de *confirmed, deaths* y *recovered*, para ello se transforman en la estructura de datos: Data Frame propia de la libreria de pandas para una mejor visualizacion, manipulacion y tratamiento de los datos.

In [ ]:
#Data Frames iniciales para casos confirmados nuevos, decesos y recuperaciones
df_c = pd.DataFrame(confirmed)
df_d = pd.DataFrame(deaths)
df_r = pd.DataFrame(recovered)

# 2. Estandarizacion y Adaptacion de los Datos para los Modelos


## 2.1. Pivotado Aplicado para cada Tabla
Inicialmente es necesario conocer como estan originalmente los datos en las tablas extraidas, para el caso de las tablas que reflejan tanto los datos de casos confirmados, decesos y recuperaciones es posible notar que existe una anomalia particular en las tablas, y es que se tienen multiples columnas dedicadas exclusivamente a mostrar la cantidad de casos confirmados, muertes o recuperaciones hasta hasta una fecha especifica.

Desde un punto de vista humano es una forma de notacion sencilla para visualizar e insertar datos acumulativos, pero, en la practica, para el analisis de los datos y la adaptacion de los mismos al modelo suele ser bastante incomodo.

In [ ]:
print("Tabla de casos confirmados antes de aplicar el pivoteo:", "\n")
df_c.head()

Tabla de casos confirmados antes de aplicar el pivoteo: 



,Province/State,Country/Region,Lat,Long,1/22/20,1/23/20,1/24/20,1/25/20,1/26/20,1/27/20,...,5/20/21,5/21/21,5/22/21,5/23/21,5/24/21,5/25/21,5/26/21,5/27/21,5/28/21,5/29/21
0,unknown,afghanistan,33.93911,67.709953,0,0,0,0,0,0,...,64575,65080,65486,65728,66275,66903,67743,68366,69130,70111
1,unknown,albania,41.15330,20.168300,0,0,0,0,0,0,...,132118,132153,132176,132209,132215,132229,132244,132264,132285,132297
2,unknown,algeria,28.03390,1.659600,0,0,0,0,0,0,...,126156,126434,126651,126860,127107,127361,127646,127926,128198,128456
3,unknown,andorra,42.50630,1.521800,0,0,0,0,0,0,...,13569,13569,13569,13569,13569,13664,13671,13682,13693,13693
4,unknown,angola,-11.20270,17.873900,0,0,0,0,0,0,...,31661,31909,32149,32441,32623,32933,33338,33607,33944,34180


Por ello se aplica una tecnica de adaptacion de los datos llamada *melt* o *pivotado*, la cual permite unificar las multiples columnas con fechas especificas en una sola columna que se denominara *Date* para este caso, ademas, esta funcion permite que se generen nuevas filas por todos y cada uno de los valores que se reflejaban en las columnas eliminadas, pasando de una notacion vertical a una horizontal.

Ademas de ello, la funcion *melt* permite especificar que columnas deberan de mantenerse sin modificacion, que en este caso son las columnas comunes en todas las tablas de: **Province/State, Country/Region, Lat** y **Long**. Este proceso se aplicara para las 3 tablas establecidas como se observa a continuacion:

In [ ]:
#Pivotado aplicado a la tabla de casos confirmados:
df_c_formated = df_c.melt(
    id_vars = ["Province/State", "Country/Region", "Lat", "Long"],
    var_name = "Date",
    value_name = "Confirmed"
)

In [ ]:
print("Tabla de casos confirmados despues de aplicar el pivoteo:", "\n")
df_c_formated.head()

Tabla de casos confirmados despues de aplicar el pivoteo: 



,Province/State,Country/Region,Lat,Long,Date,Confirmed
0,unknown,afghanistan,33.93911,67.709953,1/22/20,0
1,unknown,albania,41.15330,20.168300,1/22/20,0
2,unknown,algeria,28.03390,1.659600,1/22/20,0
3,unknown,andorra,42.50630,1.521800,1/22/20,0
4,unknown,angola,-11.20270,17.873900,1/22/20,0


In [ ]:
print("Tabla de decesos registrados antes de aplicar el pivoteo:", "\n")
df_d.head()

Tabla de decesos registrados antes de aplicar el pivoteo: 



,Province/State,Country/Region,Lat,Long,1/22/20,1/23/20,1/24/20,1/25/20,1/26/20,1/27/20,...,5/20/21,5/21/21,5/22/21,5/23/21,5/24/21,5/25/21,5/26/21,5/27/21,5/28/21,5/29/21
0,unknown,afghanistan,33.93911,67.709953,0,0,0,0,0,0,...,2772,2782,2792,2802,2812,2836,2855,2869,2881,2899
1,unknown,albania,41.15330,20.168300,0,0,0,0,0,0,...,2440,2441,2442,2444,2445,2447,2447,2447,2448,2449
2,unknown,algeria,28.03390,1.659600,0,0,0,0,0,0,...,3401,3405,3411,3418,3426,3433,3440,3448,3455,3460
3,unknown,andorra,42.50630,1.521800,0,0,0,0,0,0,...,127,127,127,127,127,127,127,127,127,127
4,unknown,angola,-11.20270,17.873900,0,0,0,0,0,0,...,704,709,715,725,731,735,742,745,749,757


In [ ]:
#Pivotado aplicado a la tabla de decesos registrados:
df_d_formated = df_d.melt(
    id_vars = ["Province/State", "Country/Region", "Lat", "Long"],
    var_name = "Date",
    value_name = "Deaths"
)

In [ ]:
print("Tabla de decesos registrados despues de aplicar el pivoteo:", "\n")
df_d_formated.head()

Tabla de decesos registrados despues de aplicar el pivoteo: 



,Province/State,Country/Region,Lat,Long,Date,Deaths
0,unknown,afghanistan,33.93911,67.709953,1/22/20,0
1,unknown,albania,41.15330,20.168300,1/22/20,0
2,unknown,algeria,28.03390,1.659600,1/22/20,0
3,unknown,andorra,42.50630,1.521800,1/22/20,0
4,unknown,angola,-11.20270,17.873900,1/22/20,0


In [ ]:
print("Tabla de recuperaciones registradas antes de aplicar el pivoteo:", "\n")
df_r.head()

Tabla de recuperaciones registradas antes de aplicar el pivoteo: 



,Province/State,Country/Region,Lat,Long,1/22/20,1/23/20,1/24/20,1/25/20,1/26/20,1/27/20,...,5/20/21,5/21/21,5/22/21,5/23/21,5/24/21,5/25/21,5/26/21,5/27/21,5/28/21,5/29/21
0,unknown,afghanistan,33.93911,67.709953,0,0,0,0,0,0,...,55687,55790,55889,56035,56295,56518,56711,56962,57119,57281
1,unknown,albania,41.15330,20.168300,0,0,0,0,0,0,...,127869,128425,128601,128732,128826,128907,128978,129042,129097,129215
2,unknown,algeria,28.03390,1.659600,0,0,0,0,0,0,...,87902,88066,88208,88346,88497,88672,88861,89040,89232,89419
3,unknown,andorra,42.50630,1.521800,0,0,0,0,0,0,...,13234,13234,13234,13234,13234,13263,13381,13405,13416,13416
4,unknown,angola,-11.20270,17.873900,0,0,0,0,0,0,...,26483,26513,26775,26778,27087,27204,27467,27529,27577,27646


In [ ]:
#Pivotado aplicado a la tabla de decesos registrados:
df_r_formated = df_r.melt(
    id_vars = ["Province/State", "Country/Region", "Lat", "Long"],
    var_name = "Date",
    value_name = "Recoveries"
)

In [ ]:
print("Tabla de recuperaciones registradas despues de aplicar el pivoteo:", "\n")
df_r_formated.head()

Tabla de recuperaciones registradas despues de aplicar el pivoteo: 



,Province/State,Country/Region,Lat,Long,Date,Recoveries
0,unknown,afghanistan,33.93911,67.709953,1/22/20,0
1,unknown,albania,41.15330,20.168300,1/22/20,0
2,unknown,algeria,28.03390,1.659600,1/22/20,0
3,unknown,andorra,42.50630,1.521800,1/22/20,0
4,unknown,angola,-11.20270,17.873900,1/22/20,0


## 2.2. Ajuste de Tipos de Dato para Fechas
Tras realizar el pivotado en cada tabla, es posible apreciar que la columna generada *Date* es de tipo object sumado a que visiblemente no presenta el formato estandar que deberian de presentar sus valores al ser fechas. Con motivos de facilitar calculos posteriores como la identificacion del dia de la semana correspondiente es necesario transformar ese tipo de dato a el tipo de dato de fechas estandar que ofrece la libreria de pandas.

In [ ]:
df_c_formated.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 135356 entries, 0 to 135355
Data columns (total 6 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   Province/State  135356 non-null  object 
 1   Country/Region  135356 non-null  object 
 2   Lat             135356 non-null  float64
 3   Long            135356 non-null  float64
 4   Date            135356 non-null  object 
 5   Confirmed       135356 non-null  int64  
dtypes: float64(2), int64(1), object(3)
memory usage: 6.2+ MB


Para aplicar la conversion de tipo de dato object a tipo fecha se emplea el uso de la funcion *to_datetime* propia de pandas que recibe como parametros la coluumna de la tabla que se desea convertir y el formato especifico especificado con expresiones regulares o regex.

In [ ]:
df_c_formated["Date"] = pd.to_datetime(df_c_formated["Date"], format = "%m/%d/%y")

Tras aplicar la transformacion, se puede notar que la columna *Date* ahora presenta el tipo de dato estandar para fechas que es *datetime64[ms]*, ademas de que visiblemente el formato correspondiente especificado en la expresion regular denota la forma anio-mes-dia.

In [ ]:
df_c_formated.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 135356 entries, 0 to 135355
Data columns (total 6 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   Province/State  135356 non-null  object        
 1   Country/Region  135356 non-null  object        
 2   Lat             135356 non-null  float64       
 3   Long            135356 non-null  float64       
 4   Date            135356 non-null  datetime64[ns]
 5   Confirmed       135356 non-null  int64         
dtypes: datetime64[ns](1), float64(2), int64(1), object(2)
memory usage: 6.2+ MB


In [ ]:
print("Valores de la columna Date despues de la conversion de tipos:", "\n")
df_c_formated.head()

Valores de la columna Date despues de la conversion de tipos: 



,Province/State,Country/Region,Lat,Long,Date,Confirmed
0,unknown,afghanistan,33.93911,67.709953,2020-01-22,0
1,unknown,albania,41.15330,20.168300,2020-01-22,0
2,unknown,algeria,28.03390,1.659600,2020-01-22,0
3,unknown,andorra,42.50630,1.521800,2020-01-22,0
4,unknown,angola,-11.20270,17.873900,2020-01-22,0


El proceso de conversion de tipos de dato de fecha a la columna *Date* se acaba de aplicar solamente a la tabla de casos confirmados, sin embargo tambien es aplicable a las tablas de decesos y recuperaciones registradas, ya que tambien por resultado del pivoteo tambien cuentan con dicha columna.

In [ ]:
#Transformacion de tipo de dato object a fecha en la tabla de decesos registrados:
df_d_formated["Date"] = pd.to_datetime(df_d_formated["Date"], format = "%m/%d/%y")
#Transformacion de tipo de dato object a fecha en la tabla de recuperaciones registradas:
df_r_formated["Date"] = pd.to_datetime(df_r_formated["Date"], format = "%m/%d/%y")

In [ ]:
print("Informacion de columnas de la tabla de decesos registrados despues de la conversion de tipos:", "\n")
df_d_formated.info()

Informacion de columnas de la tabla de decesos registrados despues de la conversion de tipos: 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 135356 entries, 0 to 135355
Data columns (total 6 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   Province/State  135356 non-null  object        
 1   Country/Region  135356 non-null  object        
 2   Lat             135356 non-null  float64       
 3   Long            135356 non-null  float64       
 4   Date            135356 non-null  datetime64[ns]
 5   Deaths          135356 non-null  int64         
dtypes: datetime64[ns](1), float64(2), int64(1), object(2)
memory usage: 6.2+ MB


In [ ]:
print("Informacion de columnas de la tabla de recuperaciones registrados despues de la conversion de tipos:", "\n")
df_r_formated.info()

Informacion de columnas de la tabla de recuperaciones registrados despues de la conversion de tipos: 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 128440 entries, 0 to 128439
Data columns (total 6 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   Province/State  128440 non-null  object        
 1   Country/Region  128440 non-null  object        
 2   Lat             128440 non-null  float64       
 3   Long            128440 non-null  float64       
 4   Date            128440 non-null  datetime64[ns]
 5   Recoveries      128440 non-null  int64         
dtypes: datetime64[ns](1), float64(2), int64(1), object(2)
memory usage: 5.9+ MB


## 2.3. Confirmacion de Valores Negativos
Antes de continuar con la adecuacion de los datos, se hace una pequenia verificacion para ver si es que no existe ningun valor registrado negativo en las columnas de *Confirmed, Deaths* y *Recoveries* para las tablas de casos confirmados, decesos y recuperaciones registradas respectivamente.

Como se observa en el codigo, el conteo de esta clase de valores en casa una de las columnas mencionadas es 0, por lo que no es necesario efectuar una limpieza instantanea para tratarlos.

In [ ]:
print("Cantidad de filas con casos confirmados acumulados negativos:", df_c_formated[df_c_formated["Confirmed"] < 0].shape[0])
print("Cantidad de filas con cantidad de muertes acumuladas negativos:", df_d_formated[df_d_formated["Deaths"] < 0].shape[0])
print("Cantidad de filas con cantidad de recuperados acumulados negativos:", df_r_formated[df_r_formated["Recoveries"] < 0].shape[0])

Cantidad de filas con casos confirmados acumulados negativos: 0
Cantidad de filas con cantidad de muertes acumuladas negativos: 0
Cantidad de filas con cantidad de recuperados acumulados negativos: 0


## 2.4. Serializacion de Frecuencias para cada Tabla
Este procedimiento surge a base de que se considera que toda los valores numericos registrados son de caracter acumulado con el paso de las fechas, respondiendo a lo que serian "frecuencias acumuladas."

Esta manera de representar la aparicion de nuevos casos confirmados, decesos y recuperaciones no es de utilidad para el proceso de aprendizaje aplicando los modelos, ya que no es informacion suficiente, sobre todo porque no refleja el factor cronologico que es vital en este dataset y que le da sentido a las acumulaciones.

A continuacion se hara la explicacion detallada en base a la tabla de casos confirmados.

### 2.4.1. Analisis Detallado del Procedimiento con la Tabla de Casos Confirmados

Por ello lo mejor es simplemente indicar cuantos nuevos casos, decesos o recuperaciones respectivamente para cada tabla se dieron en una fecha (fila) especifica, para ello se agrupan las tablas por las caracteristicas comunes de **Country/Region** y **Province/State**.

Luego se crea una nueva columna, para el caso de la tabla de casos confirmados la columna *New Cases*, que contendra la diferencia de los valores de la columna de casos confirmados acumulados desde el inicio (**Confirmed**) y la misma columna desfasada una fecha o dia atras (funcion *diff*) dando como resultado valores no acumulados, variados y serializados.

In [ ]:
df_c_formated = df_c_formated.sort_values(["Country/Region", "Province/State"])
df_c_formated["New Cases"] = df_c_formated.groupby(["Country/Region", "Province/State"])["Confirmed"].diff()

In [ ]:
print("Tabla de casos confirmados con la columna serializada agregada:", "\n")
df_c_formated.head()

Tabla de casos confirmados con la columna serializada agregada: 



,Province/State,Country/Region,Lat,Long,Date,Confirmed,New Cases
0,unknown,afghanistan,33.93911,67.709953,2020-01-22,0,NaN
274,unknown,afghanistan,33.93911,67.709953,2020-01-23,0,0.0
548,unknown,afghanistan,33.93911,67.709953,2020-01-24,0,0.0
822,unknown,afghanistan,33.93911,67.709953,2020-01-25,0,0.0
1096,unknown,afghanistan,33.93911,67.709953,2020-01-26,0,0.0


Pese a haber obtenido valores no acumulados, la serializacion presenta un problema, y es que algunos de los valores no acumulados terminan siendo negativos, que para este caso en concreto, representan que de una fecha a otra los valores de la columna **Confirmed** en lugar de haber un sufrido aumento acumulativo coherente de casos confirmados hubo decremento, cosa que es inconsistente al ser un valor que solo acumula e inevitablemente por registrar casos nuevos desde el primer dia como se observa:

In [ ]:
negative_rows = sum(map(lambda val: 1 if (val < 0) else 0, df_c_formated["New Cases"]))
print("Cantidad de filas con casos nuevos negativos:", negative_rows, "\n")
df_c_formated[df_c_formated["New Cases"] < 0]

Cantidad de filas con casos nuevos negativos: 138 



,Province/State,Country/Region,Lat,Long,Date,Confirmed,New Cases
44667,unknown,antigua and barbuda,17.060800,-61.796400,2020-07-03,68,-1.0
129607,unknown,antigua and barbuda,17.060800,-61.796400,2021-05-09,1231,-1.0
22476,australian capital territory,australia,-35.473500,149.012400,2020-04-13,102,-1.0
27957,new south wales,australia,-33.868800,151.209300,2020-05-03,3033,-2.0
34259,new south wales,australia,-33.868800,151.209300,2020-05-26,3089,-3.0
...,...,...,...,...,...,...,...
121646,unknown,united kingdom,55.378100,-3.436000,2021-04-09,4365461,-4860.0
132332,unknown,united kingdom,55.378100,-3.436000,2021-05-18,4450392,-2364.0
22459,unknown,uruguay,-32.522800,-55.765800,2020-04-12,480,-21.0
55619,unknown,yemen,15.552727,48.516388,2020-08-11,1831,-1.0


Las filas con valores no acumulados negativos simplemente no pueden ser eliminados, ya que de hacerlo se puede romper la continuidad cronologica de los registros de la tabla.

La estrategia por tanto, sera sustituir aquellos valores acumulados de la columna **Confirmed** que tras aplicar la funcion *diff* generan valores negativos por otros valores positivos, los cuales no son nada mas que la ultima ocurrencia positiva de mayor valor anterior, empleando la funcion *cummax*.

Como observacion importante a considerar tambien, es que es necesario el agrupar por **Probince/State** y **Country/Region**, ya que existen casos bordes en el reemplazo de los valores acumulados provocadores de valores negativos no acumulados en los que puede que exista una ocurrencia positiva anterior no correspondiente ni al pais ni al estado al que pertenece originalmente el valor problematico, el remplazar dicho valor con un valor no correspondiente a su grupo puede generar ruido y cesgo al momento de usar los datos en el entrenamiento del modelo.

In [ ]:
df_c_formated["Confirmed"] = df_c_formated.groupby(["Country/Region", "Province/State"])["Confirmed"].cummax()

Finalmente tras haber limpiado los valores acumulados de la columna **Confirmed** que generan valores negativos no acumulados de la serializacion, lo ultimo por hacer es recalcular los valores no acumulados nuevamente aplicando la serializacion con la funcion *diff*, en teoria ya no deberia existir la posibilidad de aparicion de filas con valores no acumulados negativos.

In [ ]:
df_c_formated = df_c_formated.sort_values(["Country/Region", "Province/State"])
df_c_formated["New Cases"] = df_c_formated.groupby(["Country/Region", "Province/State"])["Confirmed"].diff()

Efectivamente, como se puede apreciar en el codigo, el ajuste de valor maximo anterior en valores acumulados problematicos de la columna **Confirmed** aplicando la funcion *cummax* garantizo la depuracion de valores negativos no acumulados de forma controlada por grupos por pais y estado.

In [ ]:
negative_rows = sum(map(lambda val: 1 if (val < 0) else 0, df_c_formated["New Cases"]))
print("Cantidad de filas con casos nuevos negativos:", negative_rows, "\n")
df_c_formated[df_c_formated["New Cases"] < 0]

Cantidad de filas con casos nuevos negativos: 0 



,Province/State,Country/Region,Lat,Long,Date,Confirmed,New Cases


### 2.4.2. Serializacion y Control de Valores Negativos en Tablas de Decesos y Recuperaciones Registradas
El proceso de serializacion explicado tambien es aplicable a las tablas de decesos y recuperaciones registradas, al ser tablas con columnas parecidas, con la diferencia de que se deben producir los valores no acumulados a partir de las columnas **Deaths** y **Recoveries** respectivamente paea cada tabla.

In [ ]:
#Calculo de la columna New Deaths para valores no acumulados de nuevos decesos registrados por fecha:
df_d_formated = df_d_formated.sort_values(["Country/Region", "Province/State"])
df_d_formated["New Deaths"] = df_d_formated.groupby(["Country/Region", "Province/State"])["Deaths"].diff()

In [ ]:
print("Tabla de decesos registrados con la columna serializada agregada:", "\n")
df_d_formated.head()

Tabla de decesos registrados con la columna serializada agregada: 



,Province/State,Country/Region,Lat,Long,Date,Deaths,New Deaths
0,unknown,afghanistan,33.93911,67.709953,2020-01-22,0,NaN
274,unknown,afghanistan,33.93911,67.709953,2020-01-23,0,0.0
548,unknown,afghanistan,33.93911,67.709953,2020-01-24,0,0.0
822,unknown,afghanistan,33.93911,67.709953,2020-01-25,0,0.0
1096,unknown,afghanistan,33.93911,67.709953,2020-01-26,0,0.0


In [ ]:
negative_rows = sum(map(lambda val: 1 if (val < 0) else 0, df_d_formated["New Deaths"]))
print("Cantidad de filas con valores de nuevas muertes registradas negativos:", negative_rows, "\n")
df_d_formated[df_d_formated["New Deaths"] < 0]

Cantidad de filas con valores de nuevas muertes registradas negativos: 104 



,Province/State,Country/Region,Lat,Long,Date,Deaths,New Deaths
71244,unknown,angola,-11.202700,17.873900,2020-10-08,208,-3.0
35905,queensland,australia,-27.469800,153.025100,2020-06-01,6,-1.0
49610,unknown,austria,47.516200,14.550100,2020-07-21,710,-1.0
72078,unknown,austria,47.516200,14.550100,2020-10-11,851,-1.0
59481,unknown,belgium,50.833300,4.469936,2020-08-26,9879,-117.0
...,...,...,...,...,...,...,...
28753,channel islands,united kingdom,49.372300,-2.364400,2020-05-05,40,-1.0
41905,channel islands,united kingdom,49.372300,-2.364400,2020-06-22,47,-1.0
16702,"saint helena, ascension and tristan da cunha",united kingdom,-7.946700,-14.355900,2020-03-22,0,-1.0
27668,unknown,venezuela,6.423800,-66.589700,2020-05-01,10,-6.0


In [ ]:
#Reemplazo de valores acumulados de la columna Deaths que provocan valores negativos no acumulados:
df_d_formated["Deaths"] = df_d_formated.groupby(["Country/Region", "Province/State"])["Deaths"].cummax()

In [ ]:
#Recalculo de la columna serializada New Deaths para los valores no acumulados tras el ajuste:
df_d_formated = df_d_formated.sort_values(["Country/Region", "Province/State"])
df_d_formated["New Deaths"] = df_d_formated.groupby(["Country/Region", "Province/State"])["Deaths"].diff()

In [ ]:
negative_rows = sum(map(lambda val: 1 if (val < 0) else 0, df_d_formated["New Deaths"]))
print("Cantidad de filas con valores de nuevas muertes registradas negativos:", negative_rows, "\n")
df_d_formated[df_d_formated["New Deaths"] < 0]

Cantidad de filas con valores de nuevas muertes registradas negativos: 0 



,Province/State,Country/Region,Lat,Long,Date,Deaths,New Deaths


In [ ]:
#Calculo de la columna New Recoveries para valores no acumulados de nuevas recuperaciones registrados por fecha:
df_r_formated = df_r_formated.sort_values(["Country/Region", "Province/State"])
df_r_formated["New Recoveries"] = df_r_formated.groupby(["Country/Region", "Province/State"])["Recoveries"].diff()

In [ ]:
print("Tabla de recuperaciones registrados con la columna serializada agregada:", "\n")
df_r_formated.head()

Tabla de recuperaciones registrados con la columna serializada agregada: 



,Province/State,Country/Region,Lat,Long,Date,Recoveries,New Recoveries
0,unknown,afghanistan,33.93911,67.709953,2020-01-22,0,NaN
260,unknown,afghanistan,33.93911,67.709953,2020-01-23,0,0.0
520,unknown,afghanistan,33.93911,67.709953,2020-01-24,0,0.0
780,unknown,afghanistan,33.93911,67.709953,2020-01-25,0,0.0
1040,unknown,afghanistan,33.93911,67.709953,2020-01-26,0,0.0


In [ ]:
negative_rows = sum(map(lambda val: 1 if (val < 0) else 0, df_r_formated["New Recoveries"]))
print("Cantidad de filas con valores de nuevas recuperaciones registradas negativos:", negative_rows, "\n")
df_r_formated[df_r_formated["New Recoveries"] < 0]

Cantidad de filas con valores de nuevas recuperaciones registradas negativos: 226 



,Province/State,Country/Region,Lat,Long,Date,Recoveries,New Recoveries
108160,unknown,afghanistan,33.939110,67.709953,2021-03-13,49471,-28.0
16122,unknown,algeria,28.033900,1.659600,2020-03-24,24,-41.0
16642,unknown,algeria,28.033900,1.659600,2020-03-26,29,-36.0
32242,unknown,algeria,28.033900,1.659600,2020-05-25,4747,-37.0
13263,unknown,andorra,42.506300,1.521800,2020-03-13,0,-1.0
...,...,...,...,...,...,...,...
25475,unknown,vietnam,14.058324,108.277199,2020-04-28,222,-3.0
25995,unknown,vietnam,14.058324,108.277199,2020-04-30,219,-3.0
53035,unknown,vietnam,14.058324,108.277199,2020-08-12,409,-42.0
19756,unknown,west bank and gaza,31.952200,35.233200,2020-04-06,24,-1.0


In [ ]:
#Reemplazo de valores acumulados de la columna Recoveries que provocan valores negativos no acumulados:
df_r_formated["Recoveries"] = df_r_formated.groupby(["Country/Region", "Province/State"])["Recoveries"].cummax()

In [ ]:
#Recalculo de la columna serializada New Recoveries para los valores no acumulados tras el ajuste:
df_r_formated = df_r_formated.sort_values(["Country/Region", "Province/State"])
df_r_formated["New Recoveries"] = df_r_formated.groupby(["Country/Region", "Province/State"])["Recoveries"].diff()

In [ ]:
negative_rows = sum(map(lambda val: 1 if (val < 0) else 0, df_r_formated["New Recoveries"]))
print("Cantidad de filas con valores de nuevas recuperaciones registradas negativos:", negative_rows, "\n")
df_r_formated[df_r_formated["New Recoveries"] < 0]

Cantidad de filas con valores de nuevas recuperaciones registradas negativos: 0 



,Province/State,Country/Region,Lat,Long,Date,Recoveries,New Recoveries


## 2.5. Especificacion de Datos y Agregado de Caracteristicas
Los datos utilizados hasta ahora contemplan una gran variedad de paises entre los registros, esa variedad de cierta manera puede llegar a complicar el proposito del modelo y tambien el proceso del aprendizaje.

Es por ello que se decidio delimitar un poco mas los alcances y contexto de la tarea que debe cumplir el modelo, para ello se hace una revision de que pais resulta el mas recurrente y frecuente en apariciones en todos los registros, al aparecer mas veces puede resultar interesante para analizarlo mas ampliamente.

Cabe recalcar que para el presente entregable es importante aclarar que para el alcance de los modelos a entrebar unicamente se contemplara el aprendizaje y prediccion de cantidades de nuevos casos confirmados, no se descarta implementar posteriormente modelos para predecir la cantidad de decesos y recuperaciones.

In [ ]:
print(df_c_formated["Country/Region"].value_counts(), '\n')
print(df_d_formated["Country/Region"].value_counts(), '\n')
print(df_r_formated["Country/Region"].value_counts(), "\n")

Country/Region
china                 16302
canada                 7410
france                 5928
united kingdom         5928
australia              3952
                      ...  
vietnam                 494
west bank and gaza      494
yemen                   494
zambia                  494
zimbabwe                494
Name: count, Length: 193, dtype: int64 

Country/Region
china                 16302
canada                 7410
france                 5928
united kingdom         5928
australia              3952
                      ...  
vietnam                 494
west bank and gaza      494
yemen                   494
zambia                  494
zimbabwe                494
Name: count, Length: 193, dtype: int64 

Country/Region
china                 16302
france                 5928
united kingdom         5928
australia              3952
netherlands            2470
                      ...  
vietnam                 494
west bank and gaza      494
yemen                   494
zambi

### 2.5.1. Especificacion de Datos
Puesto que se pretende considerar unicamente el pais, se optara por reducir las tablas a columnas comunes **Country/Region** y **Date**, y las columnas con valores acumulados **Confirmed, Deaths** y **Recoveries** respectivamente para cada tabla. Se excluye la columna **Province/State** ya que la idea es considerar todos los casos del pais como grupo de forma simultanea.

Ademas de esto, a traves de la funcion *sum* se adicionaran todos los valores acumulados agrupando los registros segun un mismo pais y una misma fecha, generalizando los valores acumulados solo a cada pais, ademas de usar la funcion *reset_index* para reenumerar los registros tras unificarlos por Pais y Fecha.

In [ ]:
#Reduccion de la tabla de casos confirmados a sus columnas escenciales de Pais, Fecha y Cantidad de casos confirmados acumulados
df_confirmed_country = df_c_formated.groupby(["Country/Region", "Date"])["Confirmed"].sum().reset_index()

In [ ]:
print("Tabla de casos confirmados acumulados agrupados por Pais y Fecha:", "\n")
df_confirmed_country.head()

Tabla de casos confirmados acumulados agrupados por Pais y Fecha: 



,Country/Region,Date,Confirmed
0,afghanistan,2020-01-22,0
1,afghanistan,2020-01-23,0
2,afghanistan,2020-01-24,0
3,afghanistan,2020-01-25,0
4,afghanistan,2020-01-26,0


In [ ]:
#Reduccion de la tabla de decesos registrados a sus columnas escenciales de Pais, Fecha y Cantidad de decesos registrados acumulados
df_deaths_country = df_d_formated.groupby(["Country/Region", "Date"])["Deaths"].sum().reset_index()
print("Tabla de decesos registrados acumulados agrupados por Pais y Fecha:", "\n")
df_deaths_country.head()

Tabla de decesos registrados acumulados agrupados por Pais y Fecha: 



,Country/Region,Date,Deaths
0,afghanistan,2020-01-22,0
1,afghanistan,2020-01-23,0
2,afghanistan,2020-01-24,0
3,afghanistan,2020-01-25,0
4,afghanistan,2020-01-26,0


In [ ]:
#Reduccion de la tabla de recuperaciones registradas a sus columnas escenciales de Pais, Fecha y Cantidad de decesos registrados acumulados
df_recoveries_country = df_r_formated.groupby(["Country/Region", "Date"])["Recoveries"].sum().reset_index()
print("Tabla de recuperaciones registradas acumulados agrupados por Pais y Fecha:", "\n")
df_recoveries_country.head()

Tabla de recuperaciones registradas acumulados agrupados por Pais y Fecha: 



,Country/Region,Date,Recoveries
0,afghanistan,2020-01-22,0
1,afghanistan,2020-01-23,0
2,afghanistan,2020-01-24,0
3,afghanistan,2020-01-25,0
4,afghanistan,2020-01-26,0


Ademas de esto se tomo como decision general unificar o fusionar las tablas de casos confirmados, decesos y recupreaciones registradas, al ser tablas con columnas similares, deja de terner mucho sentido analizarlas por separado.

Haciendo uso de la funcion *merge* primero uniendo las tablas de casos confirmados y decesos, y finalmente esa union con la tabla de recuperaciones, siempre especificando aquellas columnas comunes entre tablas para no sobreescribir (**Country/Region** y **Date**).

In [ ]:
df = df_confirmed_country.merge(df_deaths_country, on = ["Country/Region", "Date"])
df = df.merge(df_recoveries_country, on = ["Country/Region", "Date"])
print("Tabla unificada de casos confirmados, decesos y recuperaciones registradas:", "\n")
df.head()

Tabla unificada de casos confirmados, decesos y recuperaciones registradas: 



,Country/Region,Date,Confirmed,Deaths,Recoveries
0,afghanistan,2020-01-22,0,0,0
1,afghanistan,2020-01-23,0,0,0
2,afghanistan,2020-01-24,0,0,0
3,afghanistan,2020-01-25,0,0,0
4,afghanistan,2020-01-26,0,0,0


El pais con mayor cantidad de ocurrencias a lo largo del dataset resulta ser China, por lo tanto, el modelo se especializara en el aprendizaje de datos para la prediccion de casos confirmados de COVID en dicho pais, para ello se consultan solo los registros de China y se resetea el conteo de indice de cada registro desde 0 con *reset_index*.

In [ ]:
df_china = df[df["Country/Region"] == "china"].copy().reset_index(drop = True)
print("Datos de casos confirmados, decesos y recuperaciones en China:", "\n")
df_china.head()

Datos de casos confirmados, decesos y recuperaciones en China: 



,Country/Region,Date,Confirmed,Deaths,Recoveries
0,china,2020-01-22,548,17,28
1,china,2020-01-23,643,18,30
2,china,2020-01-24,920,26,36
3,china,2020-01-25,1406,42,39
4,china,2020-01-26,2075,56,49


### 2.5.2. Agregado de Caracteristicas
Es necesario volver a hacer la serializacion de los datos para obtener valores no acumulados representados por la columna **New Cases** a partir de la columna de valores acumulados de casos confirmados **Confirmed**, la razon del porque no se conservaron las columnas de misma naturaleza agregadas en pasos anteriores se debe a que dichos valores acumulados presentaban un sesgo al ser dependientes tanto del Pais (**Country/Region**) como de las Provincias (**Province/State**), estos nuevos valores dependeran exclusivamente del Pais y la Fecha.

Otra observacion interesante es la inexistencia de casos bordes, al haverse hecho la depuracion de valores acumulados problematicos en pasos anteriores, pese a existir dependencia a la columna **Province/State**, ya no existe probabilidad de generacion de valores acumulados negativos, a parte de que tambien la serializacion se estara haciendo exclusivamente a nivel pais.

In [ ]:
df_china = df_china.sort_values(["Country/Region", "Date"])
df_china["New Cases"] = df_china.groupby(["Country/Region"])["Confirmed"].diff()

In [ ]:
print("Tabla de datos unificada con la columna New Cases con valores no acumulados:", "\n")
df_china.head()

Tabla de datos unificada con la columna New Cases con valores no acumulados: 



,Country/Region,Date,Confirmed,Deaths,Recoveries,New Cases
0,china,2020-01-22,548,17,28,NaN
1,china,2020-01-23,643,18,30,95.0
2,china,2020-01-24,920,26,36,277.0
3,china,2020-01-25,1406,42,39,486.0
4,china,2020-01-26,2075,56,49,669.0


In [ ]:
df_china["New Deaths"] = df_china.groupby(["Country/Region"])["Deaths"].diff()
df_china["New Recoveries"] = df_china.groupby(["Country/Region"])["Recoveries"].diff()
print("Tabla de datos unificada con las columnas New Cases, New Deaths y New Recoveries con valores no acumulados:", "\n")

df_china.head()

Tabla de datos unificada con las columnas New Cases, New Deaths y New Recoveries con valores no acumulados: 



,Country/Region,Date,Confirmed,Deaths,Recoveries,New Cases,New Deaths,New Recoveries
0,china,2020-01-22,548,17,28,NaN,NaN,NaN
1,china,2020-01-23,643,18,30,95.0,1.0,2.0
2,china,2020-01-24,920,26,36,277.0,8.0,6.0
3,china,2020-01-25,1406,42,39,486.0,16.0,3.0
4,china,2020-01-26,2075,56,49,669.0,14.0,10.0


Aqui se opta por reorganizar las columnas para facilitar su manejo

In [ ]:
df_china = df_china[["Country/Region", "Date", "New Cases", "Confirmed", "New Deaths", "Deaths", "New Recoveries", "Recoveries"]]
df_china.head()

,Country/Region,Date,New Cases,Confirmed,New Deaths,Deaths,New Recoveries,Recoveries
0,china,2020-01-22,NaN,548,NaN,17,NaN,28
1,china,2020-01-23,95.0,643,1.0,18,2.0,30
2,china,2020-01-24,277.0,920,8.0,26,6.0,36
3,china,2020-01-25,486.0,1406,16.0,42,3.0,39
4,china,2020-01-26,669.0,2075,14.0,56,10.0,49


A partir de este punto se comienza con la inclusion de nuevos grupos de caracteristicas, el primero se trata de 4 caracteristicas que referencian la cantidad de casos confirmados hace 2, 3, 5 y 7 dias respecto una fecha (valores que corresponden normalmente a la columna **New Cases**), estos valores sirven como referencia y sobre todo como una "memoria" que ayudara al modelo durante el proceso de aprendizaje a situarse temporalmente.

In [ ]:
#Creacion de columnas de referencia temporal de casos conformados hace 2, 3, 5 y 7 dias respcto a una fecha.
shift_days = [2, 3, 5, 7]
for sd in shift_days:
  name = f"NC-{sd}d-ago"
  col = df_china["New Cases"].shift(sd)
  df_china.insert(2, name, col)
df_china.head()

,Country/Region,Date,NC-7d-ago,NC-5d-ago,NC-3d-ago,NC-2d-ago,New Cases,Confirmed,New Deaths,Deaths,New Recoveries,Recoveries
0,china,2020-01-22,NaN,NaN,NaN,NaN,NaN,548,NaN,17,NaN,28
1,china,2020-01-23,NaN,NaN,NaN,NaN,95.0,643,1.0,18,2.0,30
2,china,2020-01-24,NaN,NaN,NaN,NaN,277.0,920,8.0,26,6.0,36
3,china,2020-01-25,NaN,NaN,NaN,95.0,486.0,1406,16.0,42,3.0,39
4,china,2020-01-26,NaN,NaN,95.0,277.0,669.0,2075,14.0,56,10.0,49


Estas 4 caracteristicas historicas de 2, 3, 5 y 7 dias atras tambien pueden aplicarse respecto a los valores de **New Deaths** y **New Recoveries** para que el modelo tenga referencia temporal sobre la cantidad de decesos y recuperaciones registradas en dias anteriores.

In [ ]:
#Creacion de columnas de referencia temporal de decesos registrados hace 2, 3, 5 y 7 dias respcto a una fecha.
for sd in shift_days:
  name = f"ND-{sd}d-ago"
  col = df_china["New Deaths"].shift(sd)
  df_china.insert(8, name, col)
df_china.head()

,Country/Region,Date,NC-7d-ago,NC-5d-ago,NC-3d-ago,NC-2d-ago,New Cases,Confirmed,ND-7d-ago,ND-5d-ago,ND-3d-ago,ND-2d-ago,New Deaths,Deaths,New Recoveries,Recoveries
0,china,2020-01-22,NaN,NaN,NaN,NaN,NaN,548,NaN,NaN,NaN,NaN,NaN,17,NaN,28
1,china,2020-01-23,NaN,NaN,NaN,NaN,95.0,643,NaN,NaN,NaN,NaN,1.0,18,2.0,30
2,china,2020-01-24,NaN,NaN,NaN,NaN,277.0,920,NaN,NaN,NaN,NaN,8.0,26,6.0,36
3,china,2020-01-25,NaN,NaN,NaN,95.0,486.0,1406,NaN,NaN,NaN,1.0,16.0,42,3.0,39
4,china,2020-01-26,NaN,NaN,95.0,277.0,669.0,2075,NaN,NaN,1.0,8.0,14.0,56,10.0,49


In [ ]:
#Creacion de columnas de referencia temporal de recuperaciones registrados hace 2, 3, 5 y 7 dias respcto a una fecha.
for sd in shift_days:
  name = f"NR-{sd}d-ago"
  col = df_china["New Recoveries"].shift(sd)
  df_china.insert(14, name, col)
df_china.head()

,Country/Region,Date,NC-7d-ago,NC-5d-ago,NC-3d-ago,NC-2d-ago,New Cases,Confirmed,ND-7d-ago,ND-5d-ago,ND-3d-ago,ND-2d-ago,New Deaths,Deaths,NR-7d-ago,NR-5d-ago,NR-3d-ago,NR-2d-ago,New Recoveries,Recoveries
0,china,2020-01-22,NaN,NaN,NaN,NaN,NaN,548,NaN,NaN,NaN,NaN,NaN,17,NaN,NaN,NaN,NaN,NaN,28
1,china,2020-01-23,NaN,NaN,NaN,NaN,95.0,643,NaN,NaN,NaN,NaN,1.0,18,NaN,NaN,NaN,NaN,2.0,30
2,china,2020-01-24,NaN,NaN,NaN,NaN,277.0,920,NaN,NaN,NaN,NaN,8.0,26,NaN,NaN,NaN,NaN,6.0,36
3,china,2020-01-25,NaN,NaN,NaN,95.0,486.0,1406,NaN,NaN,NaN,1.0,16.0,42,NaN,NaN,NaN,2.0,3.0,39
4,china,2020-01-26,NaN,NaN,95.0,277.0,669.0,2075,NaN,NaN,1.0,8.0,14.0,56,NaN,NaN,2.0,6.0,10.0,49


Otro grupo importante de caracteristicas que se considero agregar es la media de a la cantidad de casos confirmados, decesos y recuperaciones registradas desde hace una semana respecto a una determinada fecha, esta caracteristica al presentar un valor que es una medida de tendencia central puede servir como pauta para el aprendizaje del modelo para identificar aproximadamente cuan fuerte ha sido el aumento o decaida de la aparicion de casos nuevos, decesos nuevos o recuperaciones nuevas.

In [ ]:
#Creacion de la columna de referencia de la media de cantidad de casos confirmados registrados desde hace 1 semana respcto a una fecha.
columns = list(df_china.keys())
i = columns.index("Confirmed") + 1
name = "NC-7d-mean"
col = df_china["New Cases"].shift(1).rolling(7).mean()
df_china.insert(i, name, col)

In [ ]:
print("Datos con la columna de referencia de la media de cantidad de casos confirmados registrados desde hace 1 semana respcto a una fecha:", "\n")
df_china.head()

Datos con la columna de referencia de la media de cantidad de casos confirmados registrados desde hace 1 semana respcto a una fecha: 



,Country/Region,Date,NC-7d-ago,NC-5d-ago,NC-3d-ago,NC-2d-ago,New Cases,Confirmed,NC-7d-mean,ND-7d-ago,...,ND-3d-ago,ND-2d-ago,New Deaths,Deaths,NR-7d-ago,NR-5d-ago,NR-3d-ago,NR-2d-ago,New Recoveries,Recoveries
0,china,2020-01-22,NaN,NaN,NaN,NaN,NaN,548,NaN,NaN,...,NaN,NaN,NaN,17,NaN,NaN,NaN,NaN,NaN,28
1,china,2020-01-23,NaN,NaN,NaN,NaN,95.0,643,NaN,NaN,...,NaN,NaN,1.0,18,NaN,NaN,NaN,NaN,2.0,30
2,china,2020-01-24,NaN,NaN,NaN,NaN,277.0,920,NaN,NaN,...,NaN,NaN,8.0,26,NaN,NaN,NaN,NaN,6.0,36
3,china,2020-01-25,NaN,NaN,NaN,95.0,486.0,1406,NaN,NaN,...,NaN,1.0,16.0,42,NaN,NaN,NaN,2.0,3.0,39
4,china,2020-01-26,NaN,NaN,95.0,277.0,669.0,2075,NaN,NaN,...,1.0,8.0,14.0,56,NaN,NaN,2.0,6.0,10.0,49


In [ ]:
#Creacion de la columna de referencia de la media de cantidad de decesos registrados desde hace 1 semana respcto a una fecha.
columns = list(df_china.keys())
i = columns.index("Deaths") + 1
name = "ND-7d-mean"
col = df_china["New Deaths"].shift(1).rolling(7).mean()
df_china.insert(i, name, col)

In [ ]:
#Creacion de la columna de referencia de la media de cantidad de recuperaciones registradas desde hace 1 semana respcto a una fecha.
df_china["NR-7d-mean"] = df_china["New Recoveries"].shift(1).rolling(7).mean()
print("Datos con la columna de referencia de la media de cantidad de casos confirmados, decesos y recuperaciones registradas desde hace 1 semana respcto a una fecha:", "\n")
df_china.head()

Datos con la columna de referencia de la media de cantidad de casos confirmados, decesos y recuperaciones registradas desde hace 1 semana respcto a una fecha: 



,Country/Region,Date,NC-7d-ago,NC-5d-ago,NC-3d-ago,NC-2d-ago,New Cases,Confirmed,NC-7d-mean,ND-7d-ago,...,New Deaths,Deaths,ND-7d-mean,NR-7d-ago,NR-5d-ago,NR-3d-ago,NR-2d-ago,New Recoveries,Recoveries,NR-7d-mean
0,china,2020-01-22,NaN,NaN,NaN,NaN,NaN,548,NaN,NaN,...,NaN,17,NaN,NaN,NaN,NaN,NaN,NaN,28,NaN
1,china,2020-01-23,NaN,NaN,NaN,NaN,95.0,643,NaN,NaN,...,1.0,18,NaN,NaN,NaN,NaN,NaN,2.0,30,NaN
2,china,2020-01-24,NaN,NaN,NaN,NaN,277.0,920,NaN,NaN,...,8.0,26,NaN,NaN,NaN,NaN,NaN,6.0,36,NaN
3,china,2020-01-25,NaN,NaN,NaN,95.0,486.0,1406,NaN,NaN,...,16.0,42,NaN,NaN,NaN,NaN,2.0,3.0,39,NaN
4,china,2020-01-26,NaN,NaN,95.0,277.0,669.0,2075,NaN,NaN,...,14.0,56,NaN,NaN,NaN,2.0,6.0,10.0,49,NaN


Otra caracteristica interesante a agregar es la columna **Day of the Week**, que contendra numeros enteros en un rango entre el 0 al 6, donde 0 representa domingo y 6 representa sabado y asi por cada dia de la semana, esto es posible empleando la funcion *dt.day_of_week* aplicado sobre la columna **Date** que contiene valores de fecha simplemente.

In [ ]:
col = df_china["Date"].dt.day_of_week
df_china.insert(2, "Day of the Week", col)
print("Datos incluyendo la columna Day of the Week para identificar que dia de la semana corresponde una fecha especifica:", "\n")
df_china.head()

Datos incluyendo la columna Day of the Week para identificar que dia de la semana corresponde una fecha especifica: 



,Country/Region,Date,Day of the Week,NC-7d-ago,NC-5d-ago,NC-3d-ago,NC-2d-ago,New Cases,Confirmed,NC-7d-mean,...,New Deaths,Deaths,ND-7d-mean,NR-7d-ago,NR-5d-ago,NR-3d-ago,NR-2d-ago,New Recoveries,Recoveries,NR-7d-mean
0,china,2020-01-22,2,NaN,NaN,NaN,NaN,NaN,548,NaN,...,NaN,17,NaN,NaN,NaN,NaN,NaN,NaN,28,NaN
1,china,2020-01-23,3,NaN,NaN,NaN,NaN,95.0,643,NaN,...,1.0,18,NaN,NaN,NaN,NaN,NaN,2.0,30,NaN
2,china,2020-01-24,4,NaN,NaN,NaN,NaN,277.0,920,NaN,...,8.0,26,NaN,NaN,NaN,NaN,NaN,6.0,36,NaN
3,china,2020-01-25,5,NaN,NaN,NaN,95.0,486.0,1406,NaN,...,16.0,42,NaN,NaN,NaN,NaN,2.0,3.0,39,NaN
4,china,2020-01-26,6,NaN,NaN,95.0,277.0,669.0,2075,NaN,...,14.0,56,NaN,NaN,NaN,2.0,6.0,10.0,49,NaN


Tambien se opto por agregar la caracteristica **Elapsed Time** que representara la cantidad de dias acumulados desde la fehca en la que se comenzo el registro de nuevos casos confirmados, decesos y recuperaciones, es entero y acumulativo, aporta al modelo en cuanto a la nocion del periodo o epoca exacta de la pandemia en la que esta situada una prediccion.

In [ ]:
col = (df_china["Date"] - df_china["Date"].min()).dt.days
df_china.insert(3, "Elapsed Time", col)
print("Datos incluyendo la columna Elapsed Time para identificar que la epoca o punto exacto del registro en la pandemia:", "\n")
df_china.head()

Datos incluyendo la columna Elapsed Time para identificar que la epoca o punto exacto del registro en la pandemia: 



,Country/Region,Date,Day of the Week,Elapsed Time,NC-7d-ago,NC-5d-ago,NC-3d-ago,NC-2d-ago,New Cases,Confirmed,...,New Deaths,Deaths,ND-7d-mean,NR-7d-ago,NR-5d-ago,NR-3d-ago,NR-2d-ago,New Recoveries,Recoveries,NR-7d-mean
0,china,2020-01-22,2,0,NaN,NaN,NaN,NaN,NaN,548,...,NaN,17,NaN,NaN,NaN,NaN,NaN,NaN,28,NaN
1,china,2020-01-23,3,1,NaN,NaN,NaN,NaN,95.0,643,...,1.0,18,NaN,NaN,NaN,NaN,NaN,2.0,30,NaN
2,china,2020-01-24,4,2,NaN,NaN,NaN,NaN,277.0,920,...,8.0,26,NaN,NaN,NaN,NaN,NaN,6.0,36,NaN
3,china,2020-01-25,5,3,NaN,NaN,NaN,95.0,486.0,1406,...,16.0,42,NaN,NaN,NaN,NaN,2.0,3.0,39,NaN
4,china,2020-01-26,6,4,NaN,NaN,95.0,277.0,669.0,2075,...,14.0,56,NaN,NaN,NaN,2.0,6.0,10.0,49,NaN


Finalmente para concluir con la inclusion de nuevas caracteristicas y adaptacion de los datos, debido a las caracteristicas historicas de casos confirmados, decesos y/o recuperaciones existen ciertas filas que por cada *shift* se generan valores nulos ya que se pueden dar casos como el primer registro, la caracteristica de registros de hace 7 dias no es posible por ser justamente el 1er registro, esto se soluciona rellenando con valores nulos.

Dichos valores nulos realmente no aportan valor al entrenamiento del modelo, solo generan ruido, al no haber una forma razonable o logica de salvar esos registros o filas, directamente se opta eliminarlas por completo, al ser casos que abarcan registros extremos (finales e iniciales) no representan una cantidad significativa de perdida de informacion, por tanto es una desicion razonable.

In [ ]:
print("Cantidad de filas antes de eliminar filas con datos nulos:", df_china.shape[0])
df_china = df_china.dropna()
print("Cantidad de filas despues de eliminar filas con datos nulos:", df_china.shape[0])

Cantidad de filas antes de eliminar filas con datos nulos: 494
Cantidad de filas despues de eliminar filas con datos nulos: 486


In [ ]:
print("Datos de casos confirmados, decesos y recuperaciones registradas en China:", "\n")
df_china.head()

Datos de casos confirmados, decesos y recuperaciones registradas en China: 



,Country/Region,Date,Day of the Week,Elapsed Time,NC-7d-ago,NC-5d-ago,NC-3d-ago,NC-2d-ago,New Cases,Confirmed,...,New Deaths,Deaths,ND-7d-mean,NR-7d-ago,NR-5d-ago,NR-3d-ago,NR-2d-ago,New Recoveries,Recoveries,NR-7d-mean
8,china,2020-01-30,3,8,95.0,486.0,802.0,2632.0,2054.0,8141,...,38.0,171,16.571429,2.0,3.0,9.0,43.0,15.0,135,13.142857
9,china,2020-01-31,4,9,277.0,669.0,2632.0,578.0,1661.0,9802,...,42.0,213,21.857143,6.0,10.0,43.0,19.0,79.0,214,15.000000
10,china,2020-02-01,5,10,486.0,802.0,578.0,2054.0,2089.0,11891,...,46.0,259,26.714286,3.0,9.0,19.0,15.0,61.0,275,25.428571
11,china,2020-02-02,6,11,669.0,2632.0,2054.0,1661.0,4739.0,16630,...,102.0,361,31.000000,10.0,43.0,15.0,79.0,188.0,463,33.714286
12,china,2020-02-03,0,12,802.0,578.0,1661.0,2089.0,3086.0,19716,...,64.0,425,43.571429,9.0,19.0,79.0,61.0,152.0,615,59.142857


In [ ]:
df_china.info()

<class 'pandas.core.frame.DataFrame'>
Index: 486 entries, 8 to 493
Data columns (total 25 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   Country/Region   486 non-null    object        
 1   Date             486 non-null    datetime64[ns]
 2   Day of the Week  486 non-null    int32         
 3   Elapsed Time     486 non-null    int64         
 4   NC-7d-ago        486 non-null    float64       
 5   NC-5d-ago        486 non-null    float64       
 6   NC-3d-ago        486 non-null    float64       
 7   NC-2d-ago        486 non-null    float64       
 8   New Cases        486 non-null    float64       
 9   Confirmed        486 non-null    int64         
 10  NC-7d-mean       486 non-null    float64       
 11  ND-7d-ago        486 non-null    float64       
 12  ND-5d-ago        486 non-null    float64       
 13  ND-3d-ago        486 non-null    float64       
 14  ND-2d-ago        486 non-null    float64       

Para un panorama mas visual e ilustrativo se optara por usar la libreria de google sheets para ver con mayor detalle la tabla con los datos finales de China

In [ ]:
from google.colab import sheets
sheet = sheets.InteractiveSheet(df=df_china)

https://docs.google.com/spreadsheets/d/1i4yTPSNx1hxI6fqh9Wwy1BR8F8mmxqNqZy4FM_yxi5Y/edit#gid=0


# 3. Separacion de Datos
Una vez ya adaptados y definidos los datos de forma general delimitados para el analisis de casos confirmados, decesos y recuperaciones en China, ya se puede delimitar la segregacion de los mismos en 2 grupos notables, datos de entrenamiento y datos de prueba como se vera a continuacion, ya que no es habitual que un modelo se aproveche de la totalidad de los datos exclusivamente para entrenar.

## 3.1. Datos de Entrenamiento
Por convencion general al momento de hacer analisis de datos, se acordo en usar el 80% del total de los datos exclusivamente para el entrenamiento de los modelos que se van a implementar.

Existe una particularidad en la eleccion de este grupo de datos, a diferencia de como se hace habitualmente, la seleccion de estos datos no puede hacerce bajo ningun concepto de manera aleatoria o randomica, esto se debe a que son datos completamente historicos o cronologicos, optar por una seleccion al azar puede mezclar registros correspondientes al futuro y pasado en desorden, generando inconsistencias visibles, por ello directamente se cortara secuencialmente la cantidad necesaria para cubrir el 80% del total de datos como se observa a continuacion.

In [ ]:
rows = df_china.shape[0]
print("Cantidad total de filas de la tabla final para China:", rows)
data_limit = int(rows * 0.8)
print("El 80% de las filas es:", data_limit, "filas")

Cantidad total de filas de la tabla final para China: 486
El 80% de las filas es: 388 filas


In [ ]:
df_train = df_china.iloc[:data_limit]
print("Conjunto de datos de entrenamiento - 80% de la cantidad de datos en total:", "\n")
df_train.head()

Conjunto de datos de entrenamiento - 80% de la cantidad de datos en total: 



,Country/Region,Date,Day of the Week,Elapsed Time,NC-7d-ago,NC-5d-ago,NC-3d-ago,NC-2d-ago,New Cases,Confirmed,...,New Deaths,Deaths,ND-7d-mean,NR-7d-ago,NR-5d-ago,NR-3d-ago,NR-2d-ago,New Recoveries,Recoveries,NR-7d-mean
8,china,2020-01-30,3,8,95.0,486.0,802.0,2632.0,2054.0,8141,...,38.0,171,16.571429,2.0,3.0,9.0,43.0,15.0,135,13.142857
9,china,2020-01-31,4,9,277.0,669.0,2632.0,578.0,1661.0,9802,...,42.0,213,21.857143,6.0,10.0,43.0,19.0,79.0,214,15.000000
10,china,2020-02-01,5,10,486.0,802.0,578.0,2054.0,2089.0,11891,...,46.0,259,26.714286,3.0,9.0,19.0,15.0,61.0,275,25.428571
11,china,2020-02-02,6,11,669.0,2632.0,2054.0,1661.0,4739.0,16630,...,102.0,361,31.000000,10.0,43.0,15.0,79.0,188.0,463,33.714286
12,china,2020-02-03,0,12,802.0,578.0,1661.0,2089.0,3086.0,19716,...,64.0,425,43.571429,9.0,19.0,79.0,61.0,152.0,615,59.142857


## 3.2. Datos de Prueba
Es exactamente el restante 20% de la cantidad total de datos existente, a diferencia de los datos de entrenamiento, este grupo de datos sirven como filtro para detectar si el proceso de aprendizaje del modelo ha servido para afinar o refinar la capacidad de hacer predicciones o no.

Este 20% restante de los datos tambien se selccionan secuenciamente desde el punto de corte que se dejo al seleccionar los datos de entrenamiento, con la finalidad de mantener la secuencialidad de los datos y sus caracteristicas cronologicas.

In [ ]:
df_test = df_china.iloc[data_limit:]
print("Conjunto de datos de prueba - 20% de la cantidad de datos en total:", "\n")
df_test.head()

Conjunto de datos de prueba - 20% de la cantidad de datos en total: 



,Country/Region,Date,Day of the Week,Elapsed Time,NC-7d-ago,NC-5d-ago,NC-3d-ago,NC-2d-ago,New Cases,Confirmed,...,New Deaths,Deaths,ND-7d-mean,NR-7d-ago,NR-5d-ago,NR-3d-ago,NR-2d-ago,New Recoveries,Recoveries,NR-7d-mean
396,china,2021-02-21,6,396,21.0,15.0,18.0,21.0,31.0,100760,...,0.0,4833,0.571429,87.0,79.0,74.0,75.0,60.0,96041,76.857143
397,china,2021-02-22,0,397,25.0,27.0,21.0,22.0,26.0,100786,...,0.0,4833,0.571429,68.0,85.0,75.0,70.0,49.0,96090,73.000000
398,china,2021-02-23,1,398,15.0,18.0,22.0,31.0,23.0,100809,...,0.0,4833,0.571429,79.0,74.0,70.0,60.0,39.0,96129,70.285714
399,china,2021-02-24,2,399,27.0,21.0,31.0,26.0,25.0,100834,...,1.0,4834,0.285714,85.0,75.0,60.0,49.0,129.0,96258,64.571429
400,china,2021-02-25,3,400,18.0,22.0,26.0,23.0,19.0,100853,...,0.0,4834,0.142857,74.0,70.0,49.0,39.0,56.0,96314,70.857143


## 3.3. Seleccion de Caracteristicas
Tras separar los datos en datos de entrenamiento y en datos de prueba, se procede con la seleccion de caracteristicas que se consideran relevantes para el proceso de aprendizaje de los modelos, en este caso de opta por excluir todas las caracteristicas no asociadas con la cantidad de casos confirmados, ya que no estan en los alcances de lo que se pretende que hagan los modelos.

Se incluyen todas las caracteristicas referidas a referencia de datos de casos confrmados de hace 2, 3, 5 y 7 dias para dar referencia temporal al modelo como se menciono antes, relacionado a ello se incluira tambien las caracteristicas de **Day of the Week** y **Elapsed Time** para situar al modelo en el punto o epoca especifica de la pandemia y el dia de la semana asociado a la fecha, finalmente tambien es relevante la caracteristica de la media de valores de casos confirmados semanalmente como medida de tendecia central.

In [ ]:
columns = list(df_china.keys())
confirmed_model_features = columns[2:11]
confirmed_model_features.remove("Confirmed")
confirmed_model_features.remove("New Cases")
print("Caracteristicas relevantes para el modelo de prediccion de casos confirmados:", confirmed_model_features, "\n")

Caracteristicas relevantes para el modelo de prediccion de casos confirmados: ['Day of the Week', 'Elapsed Time', 'NC-7d-ago', 'NC-5d-ago', 'NC-3d-ago', 'NC-2d-ago', 'NC-7d-mean'] 



A continuacion por cada grupo de datos, tanto para los de entrenamiento como para los de prueba se arman las columnas o caracteristicas seleccionadas que el modelo usara para poder sacar una prediccion y que serviran como las entradas o variables independientes denotadas como *X_train* para los datos de entrenamiento y *X_test* para los de prueba, por otro lado, se define tambien la caracteristica que el modelo debe de aprender a predecir que en este caso corresponde a la columna **New Cases**, ya que se busca que aprendan a predecir la cantidad de casos conformados en una fecha, corresponden a las salidas o variable dependiente del modelo, denotadas como *y_train* para los datos de entrenamiento y *y_test* para los datos de prueba.

In [ ]:
#Set de datos de entrenamiento (80%):
#Caracteristicas para el aprendizaje:
X_train = df_train[confirmed_model_features]

#Caracteristica a ser aprendida:
y_train = df_train["New Cases"]
y_train = np.log1p(y_train)

In [ ]:
#Set de datos de testing (20%):
#Caracteristicas para el aprendizaje:
X_test = df_test[confirmed_model_features]

#Caracteristica a ser aprendida:
y_test = df_test["New Cases"]

# 4. Implementacion de Modelos
A continuacion en esta seccion ya se comienza a hacer el uso y consumo de los modelos de machine learning ofrecidos por la libreria de scikit learn, entre las cuales se pueden encontrar modelos regresionales, de arbol y de Support Vector Machine que es jueto lo que se busca.

Sin embargo la eleccion de los modelos no es algo que deba de ser arbitrario ya que depende de la intencion u objetivo que quiera alcanzar con el modelo, es decir la pregunta o tarea que se desee que resuelva, existen mayormente dos clases de modelo, aquellos que clasifican datos en categorias y aquellos centrados en la prediccion de datos por mejor ajuste a una funcion, para este caso, el objetivo de estos modelos se definio como la prediccion de casos confirmados en una fecha especifica en China, por tanto esta inclinado mas a lo que es la prediccion, entonces los modelos estaran inclinados a esa consigna.

Por ello se opto por el uso de los siguientes modelos:
- **Regresion Lineal Polinomica**
- **Arboles de Deesicion Regresionales** o Decision Tree Regressor
- **Support Vector Machine Regresional** o Support Vector Regression

## 4.1. Modelo de Regresion Lineal Polinomial
Se emplea el pipeline para juntar el modelo de *LinearRegresion* y las caracteristicas polinomiales (*PolynomialFeatures*) brindados por sklearn para instanciar el modelo a traves de la variable *current_model* que sera reutilizada constantemente durante toda esta parte.

Una vez instanciado el modelo lineal, se procede con su entrenamiento por medio de la funcion *fit* que recibe dos parametros que son las caracteristicas seleccionadas para el aprendizaje (*X_train*) y la caracteristica que se busca que aprenda a predecir el modelo (*y_train*).

In [ ]:
current_model = make_pipeline(PolynomialFeatures(degree=2, include_bias=False), LinearRegression())
current_model.fit(X_train, y_train)

Pipeline(steps=[('polynomialfeatures', PolynomialFeatures(include_bias=False)),
                ('linearregression', LinearRegression())])

A continucaion se ha procedido con la escritura de datos de "juguete" no completamente reales unicamente con la finalidad de constatar que el modelo instanciado y entrenado efectivamente funciona.

In [ ]:
purpossed_data = pd.DataFrame({
    "Day of the Week": [2, 3, 4],
    "Elapsed Time": [390, 391, 392],
    "NC-7d-ago": [17, 20, 11],
    "NC-5d-ago": [20, 11, 20],
    "NC-3d-ago": [7, 20, 12],
    "NC-2d-ago": [12, 12, 16],
    "NC-7d-mean": [16.4, 15.8, 15.1],
})

Finalmente se procede con la obtencion de una prediccion del modelo, para ello se usa el metodo *predict* propio del modelo y se le pasa como parametro los datos de "juguete" sugeridos anteriormente para ver si hay resultado obtenible.

In [ ]:
log_prediction = current_model.predict(purpossed_data)
current_prediction = np.expm1(log_prediction)
print("Prediccion de casos nuevos confirmados:", current_prediction)

Prediccion de casos nuevos confirmados: [21.19982802 22.37310306 22.4627459 ]


En efecto el modelo es capaz de sacar valores de prediccion fijos, de momento no existe certeza de cuan bien estan ajustados los valores que saco, ya que eso corresponde a la fase de pruebas, sin embargo se puede decir que es funcional.

## 4.2. Modelo de Arbol de Decision Regresional (Decision Tree Regressor)
Para este caso, se reusan las variables independientes ya declaradas y se aplica una transformación logarítmica (*np.log1p*) a la variable objetivo en variables exclusivas con la finalidad de escalar los valores de los datos para mejorar la fase de entrenamiento y posteriormente la capacidad de prediccion .

In [ ]:
y_train = np.log1p(df_train["New Cases"])

 Se instancia el modelo *DecisionTreeRegressor* puro y se entrena inmediatamente con los datos procesados mediante la función *.fit()*.

In [ ]:
tree_model = DecisionTreeRegressor(random_state=42)
tree_model.fit(X_train, y_train)

DecisionTreeRegressor(random_state=42)

Se generan las predicciones en escala logarítmica para los datos de "juguete" propuestos anteriormente para el modelo de regresion, y, se utiliza *np.expm1* para revertirlas a la cantidad real de casos.

In [ ]:
log_prediction_tree = tree_model.predict(purpossed_data)
current_prediction_tree = np.expm1(log_prediction_tree)
print("Predicción de casos nuevos confirmados (Árbol de Decisión):", np.round(current_prediction_tree, 2))

Predicción de casos nuevos confirmados (Árbol de Decisión): [27. 27. 22.]


## 4.3. Modelo de SVM Regresional (Support Vector Regression)
Este modelo predice un valor numerico (no para clasifica en categorias). El hiperplano de SVR necesita las variables escaladas de forma estandarizada, por eso se agrega el *StandardScaler* de sklearn antes del modelo dentro de un nuevo pipeline que los combine.

Se Reutiliza la variable *current_model* con el nuevo pipeline (escalado + SVR) y se entrena con los mismos *X_train* y *y_train* de antes. *y_train* ya tiene aplicada la transformacion logaritmica (*log1p*) para ajustar la escala de los valores de los datos.

Ademas de la instanciacion del modelo tambien se opta por ajustar el hiperparametro C en el constructor del SVR, este hiperparametro ayuda a mejorar el ajuste de los valores de prediccion incrementando la variabilidad en la curva que describe el modelo al penalizar mas los errores, con un valor de 10 se aproxima mas y mejor a lo que sacan los otros modelos como resultado.

In [ ]:
current_model = make_pipeline(StandardScaler(), SVR(C = 10))
current_model.fit(X_train, y_train)

Pipeline(steps=[('standardscaler', StandardScaler()), ('svr', SVR(C=10))])

Para verificar el funcionamiento del modelo, se reutilizaran los datos de "juguete" anteriormente declarados.

SVR si cuenta con el metodo *predict*. Obtenemos la prediccion y deshacemos la transformacion logaritmica con *expm1* para volver a la escala original.

In [ ]:
log_prediction = current_model.predict(purpossed_data)
current_prediction = np.expm1(log_prediction)
print("Prediccion de casos nuevos confirmados (SVR):", current_prediction)

Prediccion de casos nuevos confirmados (SVR): [21.74487156 21.13161553 21.04664538]
